# 18 TF-IDF

DF - document frequency;  number of times term **t** is present in all docs

**Formula:** TF-IDF = TF * IDF

**TF** = (total term t is present in doc A / total tokens in doc A)

**IDF** = log (total document / n_documents term t is present)




## sklearn

if `smooth_id=True` (the default), the constant "1" is added to the numerator and denominator of the idf as if an extra document was seen containning every term in the collection exactly once, which prevents zero divisions: idf(t) = log [(1 + n) / (1 + df(t))] + 1

### Why putting log?

Here is the intuition: if term frequency for word 'computer' in doc1 is 10 and in doc2 is 20, we can say that doc2 is more relevant than doc1 for the word 'computer'.

However, if tf of the same word 'computer' in doc1 is 1 million and doc2 is 2 millions. At this point, no much difference in terms of relevancy anymore because they both contain a very high count for term 'computer'.

Just like Debasis's answer, adding log is to dampen the importance of term that has a high frequency, e.g. using log base 2, the count of 1 million will be reduced to 19.9.

* Can refer log graph

### Limitation
- as n increase, dimensionality sparsity increases
- no capture relationship between words
- no address OOV problem

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "Thor eating pizza, Loki is eating pizza, Ironman ate pizza already",
    "Apple is announcing new iphone tomorrow",
    "Tesla is announcing new model-3 tomorrow",
    "Google is announcing new pixel-6 tomorrow",
    "Microsoft is announcing new surface tomorrow",
    "Amazon is announcing new eco-dot tomorrow",
    "I am eating biryani and you are eating grapes",
    "something is amazing"
]

In [ ]:
v = TfidfVectorizer()
transformed_output = v.fit_transform(corpus)
v.vocabulary_

{'thor': 27,
 'eating': 11,
 'pizza': 23,
 'loki': 18,
 'is': 17,
 'ironman': 16,
 'ate': 8,
 'already': 0,
 'apple': 6,
 'announcing': 5,
 'new': 21,
 'iphone': 15,
 'tomorrow': 28,
 'tesla': 26,
 'model': 20,
 'google': 13,
 'pixel': 22,
 'microsoft': 19,
 'surface': 25,
 'amazon': 3,
 'eco': 12,
 'dot': 10,
 'am': 1,
 'biryani': 9,
 'and': 4,
 'you': 29,
 'are': 7,
 'grapes': 14,
 'something': 24,
 'amazing': 2}

In [ ]:
all_feature_names = v.get_feature_names_out()

for word in all_feature_names:
  idx = v.vocabulary_.get(word)
  print(f"{word} {v.idf_[idx]}")

already 2.504077396776274
am 2.504077396776274
amazing 2.504077396776274
amazon 2.504077396776274
and 2.504077396776274
announcing 1.4054651081081644
apple 2.504077396776274
are 2.504077396776274
ate 2.504077396776274
biryani 2.504077396776274
dot 2.504077396776274
eating 2.09861228866811
eco 2.504077396776274
google 2.504077396776274
grapes 2.504077396776274
iphone 2.504077396776274
ironman 2.504077396776274
is 1.1177830356563834
loki 2.504077396776274
microsoft 2.504077396776274
model 2.504077396776274
new 1.4054651081081644
pixel 2.504077396776274
pizza 2.504077396776274
something 2.504077396776274
surface 2.504077396776274
tesla 2.504077396776274
thor 2.504077396776274
tomorrow 1.4054651081081644
you 2.504077396776274


In [ ]:
corpus [:2]

['Thor eating pizza, Loki is eating pizza, Ironman ate pizza already',
 'Apple is announcing new iphone tomorrow']

In [ ]:
transformed_output.toarray()[:2]

array([[0.24247317, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.24247317, 0.        ,
        0.        , 0.40642288, 0.        , 0.        , 0.        ,
        0.        , 0.24247317, 0.10823643, 0.24247317, 0.        ,
        0.        , 0.        , 0.        , 0.7274195 , 0.        ,
        0.        , 0.        , 0.24247317, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.31652498, 0.5639436 , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.5639436 , 0.        , 0.25173606, 0.        , 0.        ,
        0.        , 0.31652498, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.31652498, 0.        ]])

# E-Commerce Classification

In [18]:
import pandas as pd

df = pd.read_csv("Ecommerce_data.csv")
df.head()

,Text,label
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household
1,"Contrast living Wooden Decorative Box,Painted ...",Household
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories


In [19]:
df.label.value_counts()

,count
label,
Household,6000
Electronics,6000
Clothing & Accessories,6000
Books,6000


In [20]:
target = {"Household": 0, "Electronics": 1, "Clothing & Accessories": 2, "Books": 3}

df['label_num'] = df.label.map(target)
df.label_num.value_counts()

,count
label_num,
0,6000
1,6000
2,6000
3,6000


In [21]:
df.head()

,Text,label,label_num
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household,0
1,"Contrast living Wooden Decorative Box,Painted ...",Household,0
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics,1
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories,2
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories,2


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.Text,
    df.label_num,
    test_size=0.2,
    random_state=42,
    stratify=df.label_num
)

In [23]:
print("Shape of X_train", X_train.shape)
print("Shape of X_test", X_test.shape)

Shape of X_train (19200,)
Shape of X_test (4800,)


In [24]:
y_train.value_counts()

,count
label_num,
3,4800
2,4800
1,4800
0,4800


In [26]:
y_test.value_counts()

,count
label_num,
0,1200
1,1200
2,1200
3,1200


In [27]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('knn', KNeighborsClassifier())
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95      1200
           1       0.97      0.96      0.96      1200
           2       0.98      0.98      0.98      1200
           3       0.98      0.96      0.97      1200

    accuracy                           0.97      4800
   macro avg       0.97      0.97      0.97      4800
weighted avg       0.97      0.97      0.97      4800



In [30]:
X_test[:5][23834]

'Sky Tech® High Speed External Memory Card Reader USB 3.0 Hub Brand : SKY TECH Color : Silver About Sky Tech 7 in 1(10Gbps) USB 3.0 3.1 and 3 Ports Usb Hub Combo MS/ M2/ SD/ TF Card Reader compatible with Mac book/Pc/Laptop(7 in 1 combo card reader)   (1) All In One Multi-card Reader with 3 ports USB 3.0/3.1 hub Combo.  (2)With 3 USB 3.0/3.1 port with super speed 5 Gbps.  (3)With LED indication.  (4)With Fuse to protect all devices.  (5)Compliant with USB revision 2.0.  (6)With DC5V power jack.  (7)Low power consumption, suitable for Desktop and Notebook PC.  (8)Low power consumption, suitable for Desktop and Notebook PC.  (9)USB Connectors: A- type downstream X3, B-typw upstreamX1.  (10)Card Compatibility: MS,M2,SD,TF.  (11)Multi-function: 7-in-1, 3*USB 3.1 + MS/SD/M2/TF Card Reader, 7 Slots; Please noted the hub can read only one card at each time, 2 or more card cannot be read simultaneously.'

In [29]:
y_test[:5]

,label_num
10572,0
23834,1
13988,2
10777,0
11896,1


In [31]:
y_pred[:5]

array([0, 1, 2, 0, 1])

In [32]:
from sklearn.naive_bayes import MultinomialNB

clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1200
           1       0.97      0.97      0.97      1200
           2       0.98      0.98      0.98      1200
           3       0.99      0.94      0.96      1200

    accuracy                           0.96      4800
   macro avg       0.97      0.96      0.96      4800
weighted avg       0.97      0.96      0.96      4800



In [33]:
from sklearn.ensemble import RandomForestClassifier

clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('rdm', RandomForestClassifier())
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96      1200
           1       0.98      0.97      0.98      1200
           2       0.98      0.98      0.98      1200
           3       0.97      0.98      0.98      1200

    accuracy                           0.97      4800
   macro avg       0.98      0.97      0.98      4800
weighted avg       0.98      0.97      0.98      4800



In [34]:
import spacy

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
  doc = nlp(text)

  filtered_tokens = []

  for token in doc:
    if token.is_stop or token.is_punct:
      continue
    filtered_tokens.append(token.lemma_)

  return ' '.join(filtered_tokens)

In [ ]:
df["preprocessed"] = df.Text.apply(preprocess)

In [ ]:
df.head()